# AssetIQ Data Exploration

### Dataset

- Source: UCI Machine Learning Repository
- Dataset: MetroPT-3
- Domain: Industrial predictive maintenance
- Equipment: Air Production Unit (APU)
- License: CC BY 4.0

In [1]:
#Import libraries
from pathlib import Path
import sys
import json
import hashlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

sns.set_theme(style="whitegrid")

RANDOM_STATE = 42

print("Python:", sys.version)
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Interpreter:", sys.executable)


Python: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
Pandas: 3.1.1
NumPy: 2.4.6
Interpreter: c:\Users\harri\Documents\AssetIQ\.venv\Scripts\python.exe


In [2]:
#Project Root Location

ROOT = Path.cwd().resolve()

while not (ROOT / ".git").exists():
    
    if ROOT == ROOT.parent:
        raise RuntimeError("Project root not found.")
    
    ROOT = ROOT.parent

#Directory Configuration
DATA_DIR = ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

REPORTS_DIR = ROOT / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"
METRICS_DIR = REPORTS_DIR / "metrics"

CONFIG_DIR = ROOT / "configs"

#Create output directories

for directory in [
    RAW_DIR,
    PROCESSED_DIR,
    FIGURES_DIR,
    METRICS_DIR,
    CONFIG_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)


#Dataset Configuration
DATA_PATH = RAW_DIR / "MetroPT3(AirCompressor).csv"

if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_PATH}"
    )


print("Project root:", ROOT)
print("Dataset path:", DATA_PATH)
print("Dataset exists:", DATA_PATH.exists())

print(
    f"Dataset size: "
    f"{DATA_PATH.stat().st_size / 1024**2:.2f} MiB"
)

Project root: C:\Users\harri\Documents\AssetIQ
Dataset path: C:\Users\harri\Documents\AssetIQ\data\raw\MetroPT3(AirCompressor).csv
Dataset exists: True
Dataset size: 208.19 MiB


In [3]:
#Load the dataset
df = pd.read_csv(DATA_PATH)

print(f"Total observations: {df.shape[0]:,}")
print(f"Total Columns: {df.shape[1]:,}")

memory_usage = (
    df.memory_usage(deep=True).sum() / 1024**2
)

print(f"Memory usage: {memory_usage:.2f} MiB")

display(df.head(10))

Total observations: 1,516,948
Total Columns: 17
Memory usage: 224.23 MiB


,Unnamed: 0,timestamp,TP2,TP3,H1,DV_pressure,Reservoirs,Oil_temperature,Motor_current,COMP,DV_eletric,Towers,MPG,LPS,Pressure_switch,Oil_level,Caudal_impulses
0,0,2020-02-01 00:00:00,-0.012,9.358,9.340,-0.024,9.358,53.600,0.0400,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0
1,10,2020-02-01 00:00:10,-0.014,9.348,9.332,-0.022,9.348,53.675,0.0400,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0
2,20,2020-02-01 00:00:19,-0.012,9.338,9.322,-0.022,9.338,53.600,0.0425,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0
3,30,2020-02-01 00:00:29,-0.012,9.328,9.312,-0.022,9.328,53.425,0.0400,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0
4,40,2020-02-01 00:00:39,-0.012,9.318,9.302,-0.022,9.318,53.475,0.0400,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0
5,50,2020-02-01 00:00:49,-0.012,9.306,9.290,-0.024,9.308,53.500,0.0400,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0
6,60,2020-02-01 00:00:59,-0.012,9.296,9.280,-0.024,9.298,53.375,0.0400,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0
7,70,2020-02-01 00:01:09,-0.014,9.286,9.270,-0.024,9.286,53.550,0.0400,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0
8,80,2020-02-01 00:01:19,-0.012,9.276,9.258,-0.022,9.276,53.425,0.0400,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0
9,90,2020-02-01 00:01:29,-0.012,9.264,9.248,-0.022,9.264,53.375,0.0400,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1516948 entries, 0 to 1516947
Data columns (total 17 columns):
 #   Column           Non-Null Count    Dtype  
---  ------           --------------    -----  
 0   Unnamed: 0       1516948 non-null  int64  
 1   timestamp        1516948 non-null  str    
 2   TP2              1516948 non-null  float64
 3   TP3              1516948 non-null  float64
 4   H1               1516948 non-null  float64
 5   DV_pressure      1516948 non-null  float64
 6   Reservoirs       1516948 non-null  float64
 7   Oil_temperature  1516948 non-null  float64
 8   Motor_current    1516948 non-null  float64
 9   COMP             1516948 non-null  float64
 10  DV_eletric       1516948 non-null  float64
 11  Towers           1516948 non-null  float64
 12  MPG              1516948 non-null  float64
 13  LPS              1516948 non-null  float64
 14  Pressure_switch  1516948 non-null  float64
 15  Oil_level        1516948 non-null  float64
 16  Caudal_impulses  1516948 non-

In [5]:
#Initial Data Quality
schema_report = pd.DataFrame({
    "column": df.columns,
    "data_type": df.dtypes.astype(str).values,
    "missing_values": df.isna().sum().values,
    "missing_pct": (
        df.isna().mean() * 100
    ).round(4).values,
    "unique_values": df.nunique().values
})

display(schema_report)

print("Total missing values:", df.isna().sum().sum())
print("Exact duplicate rows:", df.duplicated().sum())

schema_report.to_csv(
    METRICS_DIR / "raw_schema_report.csv",
    index=False
)


,column,data_type,missing_values,missing_pct,unique_values
0,Unnamed: 0,int64,0,0.0,1516948
1,timestamp,str,0,0.0,1516948
2,TP2,float64,0,0.0,5257
3,TP3,float64,0,0.0,3683
4,H1,float64,0,0.0,2665
5,DV_pressure,float64,0,0.0,2257
6,Reservoirs,float64,0,0.0,3682
7,Oil_temperature,float64,0,0.0,2462
8,Motor_current,float64,0,0.0,1809
9,COMP,float64,0,0.0,2


Total missing values: 0
Exact duplicate rows: 0


In [6]:
df_clean = df.copy()

original_columns = df_clean.columns.tolist()

df_clean.columns = (
    df_clean.columns
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)

assert df_clean.columns.is_unique, (
    "Duplicate standardized column names detected."
)

column_mapping = pd.DataFrame({
    "original_column": original_columns,
    "standardized_column": df_clean.columns
})

display(column_mapping)

column_mapping.to_csv(
    METRICS_DIR / "column_mapping.csv",
    index=False
)

,original_column,standardized_column
0,Unnamed: 0,unnamed_0
1,timestamp,timestamp
2,TP2,tp2
3,TP3,tp3
4,H1,h1
5,DV_pressure,dv_pressure
6,Reservoirs,reservoirs
7,Oil_temperature,oil_temperature
8,Motor_current,motor_current
9,COMP,comp


In [7]:
assert "timestamp" in df_clean.columns

df_clean["timestamp"] = pd.to_datetime(
    df_clean["timestamp"],
    errors="coerce"
)

invalid_timestamps = int(
    df_clean["timestamp"].isna().sum()
)

duplicate_timestamps = int(
    df_clean["timestamp"]
    .duplicated(keep=False)
    .sum()
)

originally_sorted = (
    df_clean["timestamp"].is_monotonic_increasing
)

print("Invalid timestamps:", invalid_timestamps)
print("Duplicate timestamp rows:", duplicate_timestamps)
print("Chronologically ordered:", originally_sorted)
print("First observation:", df_clean["timestamp"].min())
print("Last observation:", df_clean["timestamp"].max())

assert invalid_timestamps == 0, (
    "Invalid timestamps detected."
)

df_clean = (
    df_clean
    .sort_values("timestamp", kind="stable")
    .reset_index(drop=True)
)

Invalid timestamps: 0
Duplicate timestamp rows: 0
Chronologically ordered: True
First observation: 2020-02-01 00:00:00
Last observation: 2020-09-01 03:59:50


In [8]:
time_deltas = (
    df_clean["timestamp"]
    .diff()
    .dt.total_seconds()
)

positive_deltas = time_deltas[time_deltas > 0]

assert not positive_deltas.empty

median_interval = positive_deltas.median()
dominant_interval = positive_deltas.mode().iloc[0]

print("Median interval:", median_interval, "seconds")
print("Dominant interval:", dominant_interval, "seconds")
print("Sampling Frequency", 1 / dominant_interval, "Hz")

print("\nMost common sampling intervals:")

display(time_deltas.value_counts().head(15))

print("\nInterval statistics:")

display(positive_deltas.describe())

Median interval: 10.0 seconds
Dominant interval: 10.0 seconds
Sampling Frequency 0.1 Hz

Most common sampling intervals:


timestamp
10.0     1337521
9.0       128277
12.0       38321
13.0        7988
11.0        4471
21.0          10
19.0           5
22.0           4
20.0           3
17.0           3
23.0           3
14.0           3
308.0          3
307.0          2
502.0          2
Name: count, dtype: int64


Interval statistics:


count    1.516947e+06
mean     1.214122e+01
std      3.141073e+02
min      8.000000e+00
25%      1.000000e+01
50%      1.000000e+01
75%      1.000000e+01
max      1.729180e+05
Name: timestamp, dtype: float64

In [9]:
validation_summary = {
    "dataset": "MetroPT-3",
    "rows": int(df.shape[0]),
    "columns": int(df.shape[1]),
    "column_names": df_clean.columns.tolist(),
    "missing_values": int(df.isna().sum().sum()),
    "exact_duplicate_rows": int(df.duplicated().sum()),
    "invalid_timestamps": invalid_timestamps,
    "duplicate_timestamp_rows": duplicate_timestamps,
    "originally_sorted": bool(originally_sorted),
    "start_timestamp": str(df_clean["timestamp"].min()),
    "end_timestamp": str(df_clean["timestamp"].max()),
    "median_interval_seconds": float(median_interval),
    "dominant_interval_seconds": float(dominant_interval),
    "sampling_frequency_hz": float(1 / dominant_interval)
}

with open(
    METRICS_DIR / "initial_validation.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(validation_summary, file, indent=4)

print(json.dumps(validation_summary, indent=4))

{
    "dataset": "MetroPT-3",
    "rows": 1516948,
    "columns": 17,
    "column_names": [
        "unnamed_0",
        "timestamp",
        "tp2",
        "tp3",
        "h1",
        "dv_pressure",
        "reservoirs",
        "oil_temperature",
        "motor_current",
        "comp",
        "dv_eletric",
        "towers",
        "mpg",
        "lps",
        "pressure_switch",
        "oil_level",
        "caudal_impulses"
    ],
    "missing_values": 0,
    "exact_duplicate_rows": 0,
    "invalid_timestamps": 0,
    "duplicate_timestamp_rows": 0,
    "originally_sorted": true,
    "start_timestamp": "2020-02-01 00:00:00",
    "end_timestamp": "2020-09-01 03:59:50",
    "median_interval_seconds": 10.0,
    "dominant_interval_seconds": 10.0,
    "sampling_frequency_hz": 0.1
}
